# 🧠 Model Comparison: Random Forest vs LSTM
**Multi-Class Classification: Healthy vs Inter-ictal vs Seizure**

This notebook provides a detailed comparative analysis between the classical ensemble approach (Random Forest) and the deep learning approach (LSTM). It examines the performance metrics and error patterns to determine the most suitable model for clinical deployment.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set aesthetic style
sns.set(style="whitegrid")
plt.rcParams.update({'font.size': 12})

## 1. Load Evaluation Metrics
Loading the JSON files generated during the independent evaluation phases.

In [2]:
metrics_dir = "../metrics"
rf_path = os.path.join(metrics_dir, "RF_metrics.json")
lstm_path = os.path.join(metrics_dir, "LSTM_metrics.json")

if not os.path.exists(rf_path) or not os.path.exists(lstm_path):
    raise FileNotFoundError("Metrics files not found. Please run the evaluation notebooks first.")

with open(rf_path, "r") as f:
    rf_metrics = json.load(f)
    
with open(lstm_path, "r") as f:
    lstm_metrics = json.load(f)

print("Metrics loaded successfully!")

## 2. Performance Comparison Table

In [3]:
# Create a DataFrame for comparison
metrics_keys = ['accuracy', 'precision', 'recall', 'f1_score']

rf_values = [rf_metrics[k] for k in metrics_keys]
lstm_values = [lstm_metrics[k] for k in metrics_keys]

comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1-Score'],
    'Random Forest': rf_values,
    'LSTM': lstm_values
})

comparison_df['Difference (LSTM - RF)'] = comparison_df['LSTM'] - comparison_df['Random Forest']
comparison_df

## 3. Visualizing Metrics

In [4]:
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(comparison_df['Metric']))
width = 0.35

rects1 = ax.bar(x - width/2, comparison_df['Random Forest'], width, label='Random Forest', color='#2ecc71')
rects2 = ax.bar(x + width/2, comparison_df['LSTM'], width, label='LSTM', color='#3498db')

ax.set_ylabel('Scores')
ax.set_title('Model Performance Comparison across Key Metrics (Macro Avg)')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Metric'])
ax.legend(loc='lower right')
ax.set_ylim(0.85, 1.05) # Zoom in to see the differences clearly

# Add score labels on top of bars
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.4f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom')

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.show()

## 4. Matrix Error Comparison
Comparing the 3x3 confusion matrices to understand where each model makes clinical mistakes.

In [5]:
rf_cm = np.array(rf_metrics['confusion_matrix'])
lstm_cm = np.array(lstm_metrics['confusion_matrix'])
labels = ["Healthy", "Inter-ictal", "Seizure"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Random Forest CM
sns.heatmap(rf_cm, annot=True, fmt="d", cmap="Greens", ax=axes[0],
            xticklabels=labels, yticklabels=labels)
axes[0].set_title("Random Forest (Multi-Class)")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

# LSTM CM
sns.heatmap(lstm_cm, annot=True, fmt="d", cmap="Blues", ax=axes[1],
            xticklabels=labels, yticklabels=labels)
axes[1].set_title("LSTM (Multi-Class)")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.tight_layout()
plt.show()